# Sports Analytics - Football

## Configuration

### Packages

In [ ]:
import requests
import duckdb
import json
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.keyvault.secrets import SecretClient
from datetime import datetime, timezone

### Variables

In [ ]:
# Will open browser for user to log in
vault_credential = DefaultAzureCredential()
vault = SecretClient(vault_url="https://breakingsouthkv.vault.azure.net/", credential=vault_credential)

API_KEY = vault.get_secret("apisports-key").value
SP_TENANT = vault.get_secret("sp-sportsanalytics-admin-tenant").value
SP_CLIENT = vault.get_secret("sp-sportsanalytics-admin-client").value
SP_SECRET = vault.get_secret("sp-sportsanalytics-admin-secret").value
AZURE_STORAGE_CONNECTION_STRING = vault.get_secret("breakingsouthsa-connection-string").value
AZURE_SQL_CONNECTION_STRING = vault.get_secret("sportsanalytics-sql-connection-string").value

sql_credential = ClientSecretCredential(SP_TENANT, SP_CLIENT, SP_SECRET)
sql_token = sql_credential.get_token("https://database.windows.net/.default").token

container_name = "sportsanalytics"
main_folder = "football"
blob_service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
container = blob_service.get_container_client(container_name)
duckdb.sql("INSTALL azure; LOAD azure;")
duckdb.sql(f"SET azure_storage_connection_string='{AZURE_STORAGE_CONNECTION_STRING}';")

# Azure SQL (written to via the DuckDB mssql community extension, authenticated as the service principal)
sql_params = dict(p.split("=", 1) for p in AZURE_SQL_CONNECTION_STRING.split(";") if "=" in p)
SQL_SERVER = sql_params.get("Server", "").replace("tcp:", "").split(",")[0]
SQL_DATABASE = sql_params.get("Database") or sql_params.get("Initial Catalog")
duckdb.sql("INSTALL mssql FROM community; LOAD mssql;")
# SCOPE keeps this secret away from az:// blob paths; unscoped, the azure extension picks it up
# for storage access and fails with "Failed to fetch key 'account_name'"
duckdb.sql(f"""
  CREATE OR REPLACE SECRET azure_sql_sp (
    TYPE AZURE,
    PROVIDER SERVICE_PRINCIPAL,
    TENANT_ID '{SP_TENANT}',
    CLIENT_ID '{SP_CLIENT}',
    CLIENT_SECRET '{SP_SECRET}',
    SCOPE 'mssql://azuresql'
  );
""")
duckdb.sql(f"""
  ATTACH 'Server={SQL_SERVER};Database={SQL_DATABASE}' AS sportsanalytics (
    TYPE mssql,
    AZURE_SECRET 'azure_sql_sp'
  );
""")

### Parameters

In [ ]:
countries = ["england", "italy", "spain", "germany", "france"]

### Classes

In [ ]:
class Entity:
  def __init__(self, table_row: tuple):
    self.report = table_row[0]
    self.url = table_row[1]
    self.endpoint = table_row[2]
    self.headers = table_row[3]
    self.params = table_row[4]
    self.frequency = table_row[5]
    self.interval = table_row[6]
    self.is_active = table_row[7]
    self.stage = table_row[8]

### Functions

#### `config_blob`

In [ ]:
def config_blob() -> str:
    result = duckdb.sql(f"""

        SELECT
            file
        FROM
            glob('az://{container_name}/{main_folder}/api-config.csv')
        ORDER BY
            file DESC
        LIMIT
            1
            
    """).fetchone()
    return result[0]

#### `collect_entities`

In [ ]:
def collect_entities(stage: int):
    data_path = config_blob()
    entities = duckdb.sql(f"""

      SELECT
        *
      FROM
        read_csv_auto('{data_path}') AS raw
      WHERE
        raw.stage = {str(stage)}

    """).fetchall()

    # Convert the raw tuples into Entity objects
    return [Entity(row) for row in entities]

#### `download_data`

In [ ]:
def download_data(entity: Entity):
  # Configure request details
  headers = {"x-apisports-key": API_KEY}
  url = entity.url
  endpoint = entity.endpoint
  params = "?" + entity.params if entity.params is not None else ""
  # Perform the request to download the data
  raw_data = requests.get(f"{url}/{endpoint}{params}", headers=headers)
  return raw_data.json()

#### `upload_raw`

In [ ]:
def upload_raw(response, name):
  ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
  container.upload_blob(f"{main_folder}/{name}/{ts}.json", json.dumps(response), overwrite=False)
  print(f"Uploaded raw data for entity: {name} at {ts}")

#### `create_raw_table`

In [ ]:
def create_raw_table(entity: str, table_name: str, nested: bool = False):
  data_path = None
  if not nested:
    result = duckdb.sql(f"""

      SELECT
        file
      FROM
        glob('az://{container_name}/{main_folder}/{entity}/*.json')
      ORDER BY
        file DESC
      LIMIT
        1

    """).fetchone()
    data_path = result[0]
  else:
    results = duckdb.sql(f"""

      SELECT
        file
      FROM (
        SELECT
          file,
          regexp_extract(file, '{entity}/([^/]+)/[^/]+\\.json$', 1) AS subfolder,
          ROW_NUMBER() OVER (PARTITION BY subfolder ORDER BY file DESC) AS rn
        FROM
          glob('az://{container_name}/{main_folder}/{entity}/*/*.json')
        )
      WHERE
        rn = 1

    """).fetchall()
    data_path = [row[0] for row in results]

  if isinstance(data_path, list):
    paths_sql = "[" + ", ".join(f"'{p}'" for p in data_path) + "]"
  else:
    paths_sql = f"'{data_path}'"

  duckdb.sql(f"DROP TABLE IF EXISTS {table_name}; CREATE TABLE {table_name} AS SELECT * FROM read_json_auto({paths_sql})")
  print(f"Created raw table: {table_name}")

## Execution

### Stage 1

#### Data Extraction

In [ ]:
entities = collect_entities(stage=1)

for entity in entities:
  has_param = entity.params is not None
  # Check if the entity is active before processing
  if entity.is_active:
    raw_data = download_data(entity)
    nested_path = entity.params.partition("=")[2] if has_param else None
    entity_name = f"{entity.report}/{nested_path}" if has_param else entity.report
    upload_raw(raw_data["response"], entity_name)
  # Create raw table for the entity
  create_raw_table(entity.report, f"latest_{entity.report}", nested=has_param)

#### Data Modeling

##### Countries

In [ ]:
duckdb.sql(f"""

  INSERT INTO
      {container_name}.{main_folder}.countries(
      id,
      code,
      name,
      flag
    )
  SELECT
    ROW_NUMBER() OVER () AS id,
    CASE
      WHEN raw.name = 'World' THEN 'WD'
      ELSE raw.code
    END AS code,
    raw.name AS name,
    raw.flag AS flag
  FROM
    latest_countries AS raw
    
""")

##### Leagues

In [ ]:
duckdb.sql(f"""

  INSERT INTO
      {container_name}.{main_folder}.leagues(
      id,
      code,
      name,
      type,
      logo,
      country_id
    )
  SELECT
    ROW_NUMBER() OVER () AS id,
    raw.league.id AS code,
    raw.league.name AS name,
    raw.league.type AS type,
    raw.league.logo AS logo,
    c.id AS country_id
  FROM
    latest_leagues AS raw
    INNER JOIN {container_name}.{main_folder}.countries AS c ON raw.country.code = c.code
    
""")

##### Seasons

In [ ]:
duckdb.sql(f"""

  WITH unnest_seasons AS (
    SELECT
      raw.league.id AS league_id,
      unnest(json_extract(raw.seasons, '$')::json[]) AS season
    FROM
      latest_leagues AS raw
  )

  INSERT INTO {container_name}.{main_folder}.seasons(
      id,
      league_id,
      "start",
      "end",
      is_current
    )
  SELECT
    ROW_NUMBER() OVER () AS id,
    l.id AS league_id,
    CAST(unn.season.start AS DATE) AS "start",
    CAST(unn.season.end AS DATE) AS "end",
    CAST(unn.season.current AS BOOLEAN) AS is_current
  FROM
    {container_name}.{main_folder}.leagues AS l
    INNER JOIN unnest_seasons AS unn ON l.code = unn.league_id
    
""")

### Stage 2

#### Data Extraction

In [ ]:
entities = collect_entities(stage=2)

for entity in entities:
  has_param = entity.params is not None
  # Check if the entity is active before processing
  if entity.is_active:
    raw_data = download_data(entity)
    nested_path = entity.params.partition("=")[2] if has_param else None
    entity_name = f"{entity.report}/{nested_path}" if has_param else entity.report
    upload_raw(raw_data["response"], entity_name)
  # Create raw table for the entity
  create_raw_table(entity.report, f"latest_{entity.report}", nested=has_param)

#### Data Modeling

##### Venues

In [ ]:
duckdb.sql(f"""

  INSERT INTO {container_name}.{main_folder}.venues(
    id,
    name,
    city,
    capacity,
    surface,
    address,
    image
  )
  SELECT DISTINCT
    raw.venue.id AS id,
    raw.venue.name AS name,
    raw.venue.city AS city,
    raw.venue.capacity AS capacity,
    raw.venue.surface AS surface,
    raw.venue.address AS address,
    raw.venue.image AS image
  FROM
    latest_teams AS raw
  WHERE
    raw.venue.id IS NOT NULL
  QUALIFY
    ROW_NUMBER() OVER (PARTITION BY raw.venue.id ORDER BY raw.venue.id) = 1
    
""")

##### Teams

In [ ]:
duckdb.sql(f"""

  INSERT INTO {container_name}.{main_folder}.teams(
    id,
    code,
    name,
    logo,
    is_national,
    venue_id
  )
  SELECT
    raw.team.id AS id,
    raw.team.code AS code,
    raw.team.name AS name,
    raw.team.logo AS logo,
    raw.team.national AS is_national,
    raw.venue.id AS venue_id
  FROM
    latest_teams AS raw
  QUALIFY
    ROW_NUMBER() OVER (PARTITION BY raw.team.id ORDER BY raw.team.id) = 1
    
""")

##### Fixtures

In [ ]:
duckdb.sql(f"""

  INSERT INTO {container_name}.{main_folder}.fixtures(
    id,
    date_id,
    time_id,
    league_id,
    home_team_id,
    away_team_id,
    halftime_home,
    halftime_away,
    fulltime_home,
    fulltime_away,
    extratime_home,
    extratime_away,
    penalty_home,
    penalty_away
  )
  SELECT
    lm.fixture.id,
    CAST(strftime(CAST(lm.fixture.date AS TIMESTAMP), '%Y%m%d') AS INT) AS date_id,
    CAST(strftime(CAST(lm.fixture.date AS TIMESTAMP), '%H%M') AS INT) AS time_id,
    CAST(lm.league.id AS SMALLINT) AS league_id,
    CAST(lm.teams.home.id AS SMALLINT) AS home_team_id,
    CAST(lm.teams.away.id AS SMALLINT) AS away_team_id,
    CAST(lm.score.halftime.home AS SMALLINT) AS halftime_home,
    CAST(lm.score.halftime.away AS SMALLINT) AS halftime_away,
    CAST(lm.score.fulltime.home AS SMALLINT) AS fulltime_home,
    CAST(lm.score.fulltime.away AS SMALLINT) AS fulltime_away,
    CAST(lm.score.extratime.home AS SMALLINT) AS extratime_home,
    CAST(lm.score.extratime.away AS SMALLINT) AS extratime_away,
    CAST(lm.score.penalty.home AS SMALLINT) AS penalty_home,
    CAST(lm.score.penalty.away AS SMALLINT) AS penalty_away
  FROM
    latest_matches lm
    
""")

##### Event Types

In [ ]:
duckdb.sql(f"""

  INSERT INTO {container_name}.{main_folder}.event_types(
    id,
    type,
    detail,
    comments
  )
  SELECT DISTINCT
    CAST(ROW_NUMBER() OVER () AS SMALLINT) AS id,
    LOWER(le.type) AS type,
    LOWER(le.detail) AS detail,
    LOWER(le.comments) AS comments
  FROM
    latest_events le

""")

##### Events

In [ ]:
duckdb.sql(f"""

  INSERT INTO {container_name}.{main_folder}.events(
    fixture_id,
    minute_id,
    team_id,
    player_id,
    player_out_id,
    event_type_id
  )
  SELECT
    1223598 AS fixture_id,
    le.time.elapsed AS minute_id,
    le.team.id AS team_id,
    le.player.id AS player_id,
    le.assist.id AS player_out_id,
    et.id AS event_type_id
  FROM
    latest_events le
    LEFT JOIN {container_name}.{main_folder}.event_types et
      ON LOWER(le.type) = et.type
      AND COALESCE(LOWER(le.detail), '') = COALESCE(et.detail, '')
      AND COALESCE(LOWER(le.comments), '') = COALESCE(et.comments, '')

""")